# 01. Подготовка клинических данных: кардиотоксичность при КРР

Основная версия основана на исходном ноутбуке v4. Вход — локальная минимизированная
таблица `data/private/source/clinical_source.csv`. Исходные номера пациентов и
календарные даты отсутствуют; возраст уже рассчитан в предоставленной автором v4.
Исход: `event_cardiotox`, отмеченный врачом. Все клинические строки и новые выходы
сохраняются только в каталогах private/ и исключены из Git.

Последовательность: загрузка, кодирование событий, составные исходы, QTc,
числовые признаки и винзоризация, категории ФВ/стадии/терапии, описательные таблицы.
Исторические правила обработки пропусков сохранены; ограничения описаны в docs/methods.md.
Ноутбук не запускался при сборке репозитория.


## 0. Импорт библиотек

In [ ]:
from pathlib import Path
import sys, os
ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents]
             if (p / "configs/project.json").is_file()), None)
if ROOT is None:
    raise FileNotFoundError("Откройте ноутбук внутри cardio-oncology-survival-analysis")
sys.path.insert(0, str(ROOT / "src"))
from cardio_oncology.paths import project_paths
P = project_paths(ROOT)
P.run.mkdir(parents=True, exist_ok=True)
P.processed.mkdir(parents=True, exist_ok=True)
OUTPUT_DIR = P.run
os.environ.setdefault("MPLCONFIGDIR", str(OUTPUT_DIR / ".matplotlib"))
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
from datetime import datetime
from scipy.stats import mannwhitneyu, chi2_contingency

warnings.filterwarnings('ignore')
sns.set(style='whitegrid')
pd.set_option('display.max_columns', 60)
pd.set_option('display.float_format', '{:.3f}'.format)

FILE_PATH = P.source / "clinical_source.csv"

RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)

## 1. Загрузка данных

In [ ]:
df_raw = pd.read_csv(FILE_PATH)
print(f'Исходный датасет: {df_raw.shape[0]} пациентов × {df_raw.shape[1]} столбцов')
df_raw.head(3)

## 2. Технический аудит

In [ ]:
miss     = df_raw.isnull().sum()
miss_pct = (miss / len(df_raw) * 100).round(1)
audit    = pd.DataFrame({'missing_n': miss, 'missing_%': miss_pct, 'dtype': df_raw.dtypes})
audit    = audit[audit['missing_n'] > 0].sort_values('missing_%', ascending=False)
print(f'Столбцов с пропусками: {len(audit)}')
display(audit)

empty_cols_raw = [c for c in df_raw.columns if df_raw[c].isna().all()]
print(f'\nПолностью пустых столбцов: {len(empty_cols_raw)}')

space_cols = [c for c in df_raw.columns if c != c.strip()]
print(f'Столбцов с пробелами в имени: {len(space_cols)}')
for c in space_cols:
    print(f'  {repr(c)}')

cardiotox_col = 'Кардиотокс(1-Да,0-Нет)'
vc = df_raw[cardiotox_col].value_counts()
print(f'\nКардиотоксичность (event_cardiotox): Есть={vc.get(1,0)} ({vc.get(1,0)/len(df_raw)*100:.1f}%), '
      f'Нет={vc.get(0,0)} ({vc.get(0,0)/len(df_raw)*100:.1f}%)')
print('→ Баланс классов ~52/48% — undersampling не нужен.')

## 3. Нормализация имён, удаление пустых столбцов и дублей

In [ ]:
df = df_raw.copy()

# ── Шаг 1: нормализация имён столбцов ────────────────────────────────────────
df.columns = df.columns.str.strip().str.replace(r'\s+', ' ', regex=True)
print('Имена столбцов нормализованы.')

# ── Шаг 2: удаление полностью пустых столбцов ────────────────────────────────
empty_cols = [c for c in df.columns if df[c].isna().all()]
df = df.drop(columns=empty_cols)
print(f'Удалено {len(empty_cols)} пустых столбцов. Осталось: {df.shape[1]}')

if 'Stage.1' in df.columns:
    if df['Stage'].equals(df['Stage.1']):
        df = df.drop(columns=['Stage.1'])
        print('Stage.1 удалён (дубль Stage).')
    else:
        print('ВНИМАНИЕ: Stage и Stage.1 различаются!')

# ── Шаг 3: два столбца «Таргет» — оставляем основной, удаляем дублирующий ────
# Медики подтвердили: нужен только бинарный флаг бевацизумаба.
col_bev_flag = 'Таргет(1-Бев,2-Рег,3-Цет/Пан,4-Комби)'
col_tgt_dup  = 'Таргет(1-Бев,2-Рег,3-Цет/Пан,4-Комби).1'
if col_bev_flag in df.columns:
    df = df.rename(columns={col_bev_flag: 'Бевацизумаб_флаг'})
    print(f'Бевацизумаб_флаг: {df["Бевацизумаб_флаг"].value_counts().to_dict()}')
if col_tgt_dup in df.columns:
    df = df.drop(columns=[col_tgt_dup])
    print('Таргет.1 удалён.')

print(f'\nИтого столбцов: {df.shape[1]}')

## 4. Возраст без календарных дат
Возраст перенесён из исходной подготовки v4. Даты рождения, диагноза и смерти в проект не включены.

In [ ]:
# Age is carried from the supplied prepared v4 cohort, using the original row order.
df["age"] = pd.to_numeric(df["age_derived"], errors="coerce")
print("Возраст: число известных значений", int(df["age"].notna().sum()))


## 5. Стандартизация пола

In [ ]:
df['sex'] = df['пол'].map({'м': 1, 'М': 1, 'ж': 0, 'Ж': 0})
print('Пол (1=мужчина, 0=женщина):')
print(df['sex'].value_counts(dropna=False))

## 6. Временна́я переменная и статус события

**Семантика столбцов (уточнено медиками):**
- `event_cardiotox` — главный исход: кардиотоксичность (выставлена врачом вручную)
- `event_death` (Общая смертность) — смерть **без** ССО: конкурирующий риск для Fine–Gray
- `event_cv_death` (Сердечно-сосудистая смерть) — смерть **с** ССО: вторичный исход

In [ ]:
df['time_days']       = pd.to_numeric(df['Дней до кардиотоксичности 0- нет даных'], errors='coerce')
df['event_cardiotox'] = pd.to_numeric(df['Кардиотокс(1-Да,0-Нет)'],   errors='coerce').astype('Int64')

# Общая смертность = смерть БЕЗ ССО (конкурирующий риск)
df['event_death']     = pd.to_numeric(df['Общая смертность смерть'],    errors='coerce').astype('Int64')

# Сердечно-сосудистая смерть = смерть С ССО (вторичный исход)
df['event_cv_death']  = pd.to_numeric(df['Сердечно-сосудистая смерть'], errors='coerce').astype('Int64')

# ── competing_status для Fine–Gray ───────────────────────────────────────────
# 0 = цензура (жив без ССО)
# 1 = ССО (event_cardiotox = 1) — основное событие
# 2 = смерть без ССО (event_death = 1) — конкурирующее событие
def competing_status(row):
    if row['event_cardiotox'] == 1: return 1
    if row['event_death']     == 1: return 2
    return 0

df['competing_status'] = df.apply(competing_status, axis=1)

print('Время до события (дни):')
print(df['time_days'].describe().round(0))
print()
print('Распределение событий:')
print(f'  Кардиотоксичность (event=1):     {df["event_cardiotox"].sum()} ({df["event_cardiotox"].mean()*100:.1f}%)')
print(f'  Смерть без ССО (competing=2):    {df["event_death"].sum()} ({df["event_death"].mean()*100:.1f}%)')
print(f'  Смерть с ССО (cv_death=1):       {df["event_cv_death"].sum()} ({df["event_cv_death"].mean()*100:.1f}%)')
print()
print('competing_status:')
print(df['competing_status'].value_counts().rename({0: 'цензура', 1: 'ССО', 2: 'смерть без ССО'}))

## 7. Составные исходы ССО

In [ ]:
# Группировка индивидуальных ССО-событий в клинически значимые составные исходы
SSO_ACS          = ['ИмспST', 'ИмбпсST', 'Нестабильная стенокардия']
SSO_RHYTHM_MAJOR = ['ФП', 'ЖТ', 'ФЖ', 'полная AV-блокада']
SSO_RHYTHM_MINOR = ['Предсердные экстрасистолии', 'замедление A-V проведения',
                    'ЖЭС', 'Миграция водителя ритма', 'синусовая тахикардия', 'Брадикардия синусовая']
SSO_THROMBO_HF   = ['Декомпенсация ранее существующей СН',
                    'Системные артериальные эмболии', 'Венозные тромбо эмболии']
ALL_SSO_COLS     = SSO_ACS + SSO_RHYTHM_MAJOR + SSO_RHYTHM_MINOR + SSO_THROMBO_HF

for c in ALL_SSO_COLS:
    df[c] = pd.to_numeric(df[c], errors='coerce').fillna(0).astype(int)

df['sso_any']          = (df[ALL_SSO_COLS].sum(axis=1)          > 0).astype(int)
df['sso_acs']          = (df[SSO_ACS].sum(axis=1)               > 0).astype(int)
df['sso_rhythm_major'] = (df[SSO_RHYTHM_MAJOR].sum(axis=1)       > 0).astype(int)
df['sso_rhythm_minor'] = (df[SSO_RHYTHM_MINOR].sum(axis=1)       > 0).astype(int)
df['sso_thrombo_hf']   = (df[SSO_THROMBO_HF].sum(axis=1)        > 0).astype(int)

# Примечание: индивидуальные SSO-столбцы (ИмспST ... де ново ССЗ) НЕ включаются
# в финальную таблицу — медики подтвердили, что достаточно составных флагов.

sso_summary = pd.DataFrame({
    'Группа':  ['Любое ССО', 'ОКС', 'Значимые аритмии', 'Незначимые аритмии', 'Тромбозы/ХСН'],
    'Флаг':    ['sso_any', 'sso_acs', 'sso_rhythm_major', 'sso_rhythm_minor', 'sso_thrombo_hf'],
    'n':       [df[f].sum() for f in ['sso_any','sso_acs','sso_rhythm_major','sso_rhythm_minor','sso_thrombo_hf']],
    '%':       [(df[f].mean()*100).round(1) for f in ['sso_any','sso_acs','sso_rhythm_major','sso_rhythm_minor','sso_thrombo_hf']]
})
display(sso_summary)

print('\nЧастота отдельных ССО-событий (для справки, не входят в финальную таблицу):')
ind = df[ALL_SSO_COLS].sum().sort_values(ascending=False)
display(pd.DataFrame({'n': ind, '%': (ind/len(df)*100).round(1)}))

## 8. QTc: конвертация с→мс и ΔQTc

In [ ]:
def clean_qtc(series):
    """
    Нормализует QTc:
    - запятая → точка ('0,38' → 0.38)
    - значения < 1 → в секундах, умножаем × 1000 → мс
    - вне [250, 700] мс → физически невозможно → NaN
    """
    r = (series.astype(str).str.strip()
               .str.replace(',', '.', regex=False)
               .pipe(pd.to_numeric, errors='coerce'))
    r = r.where(r >= 1, r * 1000)
    return r.where((r >= 250) & (r <= 700)).round(0)

df['qtc_start_ms'] = clean_qtc(df['QTc начало в секундах'])
df['qtc_last_ms']  = clean_qtc(df['QTc последнее в секундах'])
df['delta_qtc_ms'] = df['qtc_last_ms'] - df['qtc_start_ms']

# Клинические пороговые флаги ΔQTc
# qtc_start_prolonged УБРАН по решению медиков
df['delta_qtc_gt30'] = (df['delta_qtc_ms'] >= 30).astype('Int64')   # настороженность
df['delta_qtc_gt60'] = (df['delta_qtc_ms'] >= 60).astype('Int64')   # коррекция терапии

# qtc_last_ms НЕ включается в финальный датасет как предиктор:
# вся информация о динамике уже в delta_qtc_ms

print(f'QTc начало (мс): {df["qtc_start_ms"].describe()[["count","mean","min","max"]].round(1).to_dict()}')
print(f'ΔQTc (мс):       {df["delta_qtc_ms"].describe()[["count","mean","min","max"]].round(1).to_dict()}')
print(f'ΔQTc ≥ 30 мс:    {df["delta_qtc_gt30"].sum()} пациентов')
print(f'ΔQTc ≥ 60 мс:    {df["delta_qtc_gt60"].sum()} пациентов')
print(f'Пропуски QTc:    {df["qtc_start_ms"].isna().sum()} / {len(df)}')

## 9. Очистка числовых предикторов

In [ ]:
def clean_numeric_ru(series, zero_as_nan=False, lower_bound=None, upper_bound=None):
    """Очистка числового столбца: запятые→точки, нечисловые→NaN, опциональные границы."""
    r = (series.astype(str).str.strip()
               .str.replace(',', '.', regex=False)
               .pipe(pd.to_numeric, errors='coerce'))
    if zero_as_nan:    r = r.replace(0, np.nan)
    if lower_bound is not None: r = r.where(r >= lower_bound)
    if upper_bound is not None: r = r.where(r <= upper_bound)
    return r

def winsorize_col(series, lower_q=0.025, upper_q=0.975):
    """Winsorizing на уровне 2.5–97.5% квантилей."""
    lo = series.quantile(lower_q)
    hi = series.quantile(upper_q)
    return series.clip(lower=lo, upper=hi)

# ── Числовые предикторы ───────────────────────────────────────────────────────
df['bmi']           = clean_numeric_ru(df['ИМТ'], zero_as_nan=True, lower_bound=10, upper_bound=60)
df['hr']            = clean_numeric_ru(df['ЧСС'], lower_bound=20, upper_bound=250)
df['esr_start']     = clean_numeric_ru(df['соэ начало'], lower_bound=1, upper_bound=120)
df['hgb_start']     = clean_numeric_ru(df['гемоглобин в начале заболевания'], lower_bound=50, upper_bound=200)
df['glucose_start'] = clean_numeric_ru(df['глюоза на момент начала з-я'], zero_as_nan=True, lower_bound=2, upper_bound=35)
df['lvef']          = clean_numeric_ru(df['ФВ %'], lower_bound=10, upper_bound=85)
df['la_size']       = clean_numeric_ru(df['Размер левого передсердия'], lower_bound=20, upper_bound=65)
df['crp_start']     = clean_numeric_ru(df['СРБ на момент начала заболевания'], lower_bound=0)
df['wbc_start']     = clean_numeric_ru(df['лейоциты на начало'], lower_bound=1, upper_bound=50)
df['chol_start']    = clean_numeric_ru(df['холестерин нач'], zero_as_nan=True, lower_bound=1, upper_bound=15)
df['n_courses']     = clean_numeric_ru(df['Курсов(кол-во)'])

# Сводка полноты
for col, label in [('bmi','ИМТ'), ('hr','ЧСС'), ('esr_start','СОЭ'), ('hgb_start','Hgb'),
                   ('glucose_start','Глюкоза'), ('lvef','ФВ ЛЖ'), ('la_size','Размер ЛП'),
                   ('crp_start','СРБ'), ('wbc_start','Лейкоциты'), ('n_courses','Курсов ХТ')]:
    n = df[col].notna().sum()
    print(f'  {label:<15} {n:>3}/{len(df)}  [{df[col].min():.1f} – {df[col].max():.1f}]')

In [ ]:
# Winsorizing воспалительных маркеров с выраженными правыми хвостами
# QTc, hr, hgb НЕ winsorize — экстремальные значения клинически значимы
WINSORIZE_COLS = ['esr_start', 'crp_start', 'wbc_start', 'glucose_start']

print('Winsorizing (квантили 2.5% / 97.5%):')
for col in WINSORIZE_COLS:
    if col in df.columns and df[col].notna().sum() > 10:
        lo_b, hi_b = df[col].min(), df[col].max()
        df[col] = winsorize_col(df[col])
        lo_a, hi_a = df[col].min(), df[col].max()
        if (lo_b != lo_a) or (hi_b != hi_a):
            print(f'  {col:<20} {lo_b:.1f}–{hi_b:.1f}  →  {lo_a:.1f}–{hi_a:.1f}')
print('Готово.')

## 10. Фракция выброса ЛЖ — категориальная переменная

По решению медиков: ФВ представляется как 3-категориальная переменная:
- `>50%` — норма
- `40–50%` — умеренно снижена  
- `<40%` — значительно снижена

In [ ]:
# ФВ ЛЖ — порядковая числовая переменная
# 0 = норма (>= 50%), 1 = умеренно снижена (40–50%), 2 = значительно снижена (<40%)
def lvef_to_num(val):
    if pd.isnull(val):
        return np.nan
    if val >= 50: return 0
    if val >= 40: return 1
    return 2

df['lvef_cat_num'] = df['lvef'].apply(lvef_to_num).astype('Int64')

print('lvef_cat_num (0=норма ≥50%, 1=средне 40-50%, 2=плохо <40%):')
print(df['lvef_cat_num'].value_counts(dropna=False).sort_index())

## 11. Кодирование категориальных переменных

In [ ]:
# ── Стадия РТК ───────────────────────────────────────────────────────────────
_stage_raw   = pd.to_numeric(df['Stage'], errors='coerce')
_stage_clean = _stage_raw.where(_stage_raw > 0)
df['stage_advanced'] = (_stage_clean >= 3).astype('Int64')
print('Поздняя стадия (III–IV):')
print(df['stage_advanced'].value_counts(dropna=False))

# ── Схема химиотерапии ────────────────────────────────────────────────────────
# Возвращена для сравнительного анализа кардиориска по схемам
# 0=нет ХТ, 1=FOLFOX, 2=FOLFIRI, 3=XELOX, 4=DeGramont, 5=Другое
SCHEME_COL    = 'Схема(1-FOLFOX,2-FOLFIRI,3-XELOX,4-DeGramont,5-Другое)'
SCHEME_LABELS = {0:'Нет ХТ', 1:'FOLFOX', 2:'FOLFIRI', 3:'XELOX', 4:'DeGramont', 5:'Другое'}
df['cht_scheme']       = pd.to_numeric(df[SCHEME_COL], errors='coerce').astype('Int64')
df['cht_scheme_label'] = df['cht_scheme'].map(SCHEME_LABELS)
print('\nСхема ХТ:')
print(df['cht_scheme_label'].value_counts(dropna=False))

# ── Платиносодержащие препараты ───────────────────────────────────────────────
df['platinum'] = pd.to_numeric(
    df['Платиносодержащие препараты 0-нет, 1-да'], errors='coerce'
).astype('Int64')
print(f'\nПлатиносодержащие: {df["platinum"].sum()} ({df["platinum"].mean()*100:.1f}%)')

# ── Бевацизумаб ───────────────────────────────────────────────────────────────
df['bevacizumab'] = pd.to_numeric(df['Бевацизумаб_флаг'], errors='coerce').astype('Int64')
print(f'Бевацизумаб:       {df["bevacizumab"].sum()} ({df["bevacizumab"].mean()*100:.1f}%)')

# ── Гистология ────────────────────────────────────────────────────────────────
df['histology'] = pd.to_numeric(df['Гистология (1-G1, 2-G2, 3-G3, 99-НД)'], errors='coerce')
df['histology'] = df['histology'].where(df['histology'] != 99)
print(f'Гистология: {df["histology"].value_counts(dropna=False).to_dict()}')

# ── ССЗ в анамнезе ────────────────────────────────────────────────────────────
df['cvd_history'] = pd.to_numeric(df['ссз в анамнезе'], errors='coerce').astype('Int64')
print(f'ССЗ в анамнезе:    {df["cvd_history"].sum()} ({df["cvd_history"].mean()*100:.1f}%)')

# ── Метастазы в печень ────────────────────────────────────────────────────────
df['mets_liver'] = pd.to_numeric(df['МТС_Печ (0-нет, 1-есть, 99-НД)'], errors='coerce')
df['mets_liver'] = df['mets_liver'].where(df['mets_liver'] != 99)
print(f'Метастазы в печень:{(df["mets_liver"]==1).sum()} ({(df["mets_liver"]==1).mean()*100:.1f}%)')

# ── Анемия ────────────────────────────────────────────────────────────────────
hgb_thresh = df['sex'].map({1: 130, 0: 120}).fillna(125)
df['anemia'] = (df['hgb_start'] < hgb_thresh).astype('Int64')
print(f'Анемия на старте:  {df["anemia"].sum()} ({df["anemia"].mean()*100:.1f}%)')


## 12. Итоговый датасет

In [ ]:
SURVIVAL_VARS = [
    'time_days',
    'event_cardiotox',
    'event_death',
    'event_cv_death',
    'competing_status',
]

SSO_FLAGS = ['sso_any', 'sso_acs', 'sso_rhythm_major', 'sso_rhythm_minor', 'sso_thrombo_hf']

PREDICTORS = [
    # Демография
    'age', 'sex',
    # Клинический статус
    'hr', 'lvef_cat_num', 'la_size',
    # Лабораторные — воспаление
    'esr_start', 'crp_start', 'wbc_start',
    # Лабораторные — метаболизм
    'bmi', 'hgb_start', 'anemia', 'glucose_start', 'chol_start',
    # QTc
    'qtc_start_ms', 'delta_qtc_ms', 'delta_qtc_gt30', 'delta_qtc_gt60',
    # Онкологический статус
    'stage_advanced', 'histology', 'mets_liver',
    # Лечение
    'cht_scheme',        # номер схемы ХТ (для моделей)
    'cht_scheme_label',  # название схемы (для графиков и таблиц)
    'platinum',          # платиносодержащие препараты
    'n_courses',         # количество курсов ХТ
    'bevacizumab',       # ключевой предиктор
    # Сопутствующие
    'cvd_history',
]

all_cols = SURVIVAL_VARS + SSO_FLAGS + PREDICTORS
all_cols  = list(dict.fromkeys(c for c in all_cols if c in df.columns))
df_final  = df[all_cols].copy()

print(f'Финальный датасет: {df_final.shape[0]} пациентов × {df_final.shape[1]} столбцов')
display(df_final.head(3))


In [ ]:
# ── Проверка пропусков в ключевых переменных ──────────────────────────────────
key_cols = ['age', 'sex', 'hr', 'bmi', 'hgb_start', 'glucose_start',
            'stage_advanced', 'n_courses', 'time_days', 'event_cardiotox']

miss_report = pd.DataFrame({
    'missing_n': df_final[key_cols].isnull().sum(),
    'missing_%': (df_final[key_cols].isnull().sum() / len(df_final) * 100).round(1)
})
print('Пропуски в ключевых предикторах:')
display(miss_report)
print(f'\nQTc: {df_final["qtc_start_ms"].isna().sum()} пропусков '
      f'({df_final["qtc_start_ms"].isna().mean()*100:.1f}%) '
      f'— анализ QTc на подвыборке n={df_final["qtc_start_ms"].notna().sum()}')

## 13. Описательная статистика

In [ ]:
# ── Таблица 1: непрерывные предикторы ────────────────────────────────────────
cont_preds = ['age', 'hr', 'bmi', 'lvef', 'la_size',
              'hgb_start', 'glucose_start', 'esr_start', 'crp_start',
              'n_courses', 'qtc_start_ms', 'delta_qtc_ms', 'time_days']
# lvef для описательной статистики используем исходную числовую
desc = df[cont_preds].describe(percentiles=[.25, .5, .75]).T
desc = desc[['count', 'mean', 'std', '25%', '50%', '75%', 'min', 'max']].round(2)
desc.columns = ['n', 'Mean', 'SD', 'Q1', 'Median', 'Q3', 'Min', 'Max']
print('Таблица 1 — Непрерывные предикторы:')
display(desc)

In [ ]:
# ── Таблица 2: сравнение ССО vs без ССО (Mann–Whitney) ───────────────────────
rows_mw = []
for col in cont_preds:
    g0 = df_final.loc[df_final['event_cardiotox'] == 0, col].dropna() if col in df_final else pd.Series(dtype=float)
    g1 = df_final.loc[df_final['event_cardiotox'] == 1, col].dropna() if col in df_final else pd.Series(dtype=float)
    # Для lvef используем исходный числовой столбец через df
    if col == 'lvef' and col not in df_final:
        g0 = df.loc[df['event_cardiotox'] == 1, col].dropna() if col in df else pd.Series(dtype=float)
    if len(g0) > 1 and len(g1) > 1:
        _, p = mannwhitneyu(g0, g1, alternative='two-sided')
        rows_mw.append({
            'Признак':          col,
            'Без ССО Med [IQR]': f'{g0.median():.1f} [{g0.quantile(.25):.1f}–{g0.quantile(.75):.1f}]',
            'ССО Med [IQR]':    f'{g1.median():.1f} [{g1.quantile(.25):.1f}–{g1.quantile(.75):.1f}]',
            'p (MW)':           f'{p:.3f}'
        })
print('Таблица 2 — Непрерывные предикторы: ССО vs без ССО (Mann–Whitney):')
display(pd.DataFrame(rows_mw))

In [ ]:
# ── Таблица 3: категориальные предикторы (χ²) ────────────────────────────────
cat_preds = ['sex', 'cvd_history', 'anemia', 'stage_advanced',
             'platinum', 'bevacizumab', 'delta_qtc_gt30', 'delta_qtc_gt60']

rows_cat = []
for col in cat_preds:
    if col not in df_final.columns:
        continue
    ct = pd.crosstab(df_final['event_cardiotox'], df_final[col])
    if ct.shape == (2, 2):
        _, p, _, _ = chi2_contingency(ct)
        n0 = int(df_final.loc[df_final['event_cardiotox']==0, col].sum())
        n1 = int(df_final.loc[df_final['event_cardiotox']==1, col].sum())
        t0 = (df_final['event_cardiotox']==0).sum()
        t1 = (df_final['event_cardiotox']==1).sum()
        rows_cat.append({'Признак (=1)': col,
                         'Без ССО n (%)': f'{n0} ({n0/t0*100:.1f}%)',
                         'ССО n (%)':     f'{n1} ({n1/t1*100:.1f}%)',
                         'p (χ²)': f'{p:.3f}'})

print('Таблица 3 — Категориальные предикторы: ССО vs без ССО (χ²):')
display(pd.DataFrame(rows_cat))

# ФВ ЛЖ — порядковая, анализируем через Mann–Whitney
g0_lvef = df_final.loc[df_final['event_cardiotox']==0, 'lvef_cat_num'].dropna()
g1_lvef = df_final.loc[df_final['event_cardiotox']==1, 'lvef_cat_num'].dropna()
_, p_lvef = mannwhitneyu(g0_lvef, g1_lvef, alternative='two-sided')
print(f'\nФВ ЛЖ (lvef_cat_num, Mann–Whitney):')
print(f'  Без ССО: медиана={g0_lvef.median():.0f} | ССО: медиана={g1_lvef.median():.0f} | p={p_lvef:.3f}')
print(f'  Распределение по группам:')
display(pd.crosstab(df_final['event_cardiotox'], df_final['lvef_cat_num'].map(
    {0:'>50 (норма)', 1:'40-50 (средне)', 2:'<40 (плохо)'}
)))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Левая панель: гистограмма времени наблюдения
axes[0].hist(df_final.loc[df_final['event_cardiotox']==1, 'time_days'],
             bins=30, alpha=0.7, color='#D65F5F', label='ССО (event=1)')
axes[0].hist(df_final.loc[df_final['event_cardiotox']==0, 'time_days'],
             bins=30, alpha=0.5, color='#4878CF', label='Цензура (event=0)')
axes[0].set_xlabel('Дни до события / цензуры')
axes[0].set_ylabel('Число пациентов')
axes[0].set_title('Распределение времени наблюдения')
axes[0].legend()

# Правая панель: боксплот времени наблюдения по схемам ХТ
df_plot = df_final.dropna(subset=['cht_scheme_label', 'time_days'])
df_plot.boxplot(column='time_days', by='cht_scheme_label', ax=axes[1])
axes[1].set_title('Время наблюдения по схемам ХТ')
axes[1].set_xlabel('Схема ХТ')
axes[1].set_ylabel('Дни')
plt.suptitle('')

plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'time_distribution.png', dpi=150, bbox_inches='tight')
plt.show()


In [ ]:
# ── Визуализация 2: QTc ───────────────────────────────────────────────────────
df_qtc = df_final.dropna(subset=['qtc_start_ms', 'delta_qtc_ms'])

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

sns.kdeplot(df_qtc.loc[df_qtc['event_cardiotox']==0, 'qtc_start_ms'],
            ax=axes[0], label='Без ССО', color='#4878CF')
sns.kdeplot(df_qtc.loc[df_qtc['event_cardiotox']==1, 'qtc_start_ms'],
            ax=axes[0], label='ССО', color='#D65F5F')
axes[0].axvline(440, color='orange', linestyle='--', alpha=0.8, label='440 мс')
axes[0].set_xlabel('QTc начало (мс)')
axes[0].set_title('QTc исходный: ССО vs без ССО')
axes[0].legend()

color_ev = df_qtc['event_cardiotox'].map({0: '#4878CF', 1: '#D65F5F'})
axes[1].scatter(df_qtc['qtc_start_ms'], df_qtc['delta_qtc_ms'],
                c=color_ev, alpha=0.5, s=30)
axes[1].axhline(30, color='orange', linestyle='--', label='ΔQTc=30 мс')
axes[1].axhline(60, color='red',    linestyle='--', label='ΔQTc=60 мс')
axes[1].set_xlabel('QTc начало (мс)')
axes[1].set_ylabel('ΔQTc (мс)')
axes[1].set_title('ΔQTc vs QTc начало (синий=нет ССО, красный=ССО)')
axes[1].legend()

plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'qtc_distribution.png', dpi=150, bbox_inches='tight')
plt.show()
print(f'QTc-подвыборка: n={len(df_qtc)} пациентов')

## 14. Сохранение

In [ ]:
# ── Основной датасет ──────────────────────────────────────────────────────────
df_final.to_csv(P.processed / "cohort.csv", index=False, encoding='utf-8-sig')
print(f'Сохранён: РТК_ССО_подготовленный_v4.csv  ({df_final.shape[0]} × {df_final.shape[1]})')

# ── Словарь переменных ────────────────────────────────────────────────────────
var_dict = pd.DataFrame([
    # ЦЕЛЕВЫЕ
    {'Переменная': 'time_days',        'Тип': 'Время',        'Описание': 'Дни до события / цензуры'},
    {'Переменная': 'event_cardiotox',  'Тип': 'Событие',      'Описание': '1=кардиотоксичность (главный исход, выставлен врачом)'},
    {'Переменная': 'event_death',      'Тип': 'Событие',      'Описание': '1=смерть БЕЗ ССО (конкурирующий риск)'},
    {'Переменная': 'event_cv_death',   'Тип': 'Событие',      'Описание': '1=смерть С ССО (вторичный исход)'},
    {'Переменная': 'competing_status', 'Тип': 'Статус',       'Описание': '0=цензура, 1=ССО, 2=смерть без ССО'},
    # ССО-флаги
    {'Переменная': 'sso_any',          'Тип': 'Исход',        'Описание': 'Любое ССО из составных групп'},
    {'Переменная': 'sso_acs',          'Тип': 'Исход',        'Описание': 'ОКС (ИМспST / ИМбпсST / НС)'},
    {'Переменная': 'sso_rhythm_major', 'Тип': 'Исход',        'Описание': 'Значимые аритмии (ФП, ЖТ, ФЖ, AV-блокада)'},
    {'Переменная': 'sso_rhythm_minor', 'Тип': 'Исход',        'Описание': 'Незначимые аритмии (экстрасистолы, тахикардия и др.)'},
    {'Переменная': 'sso_thrombo_hf',   'Тип': 'Исход',        'Описание': 'Тромбозы + декомпенсация ХСН'},
    # ПРЕДИКТОРЫ
    {'Переменная': 'age',              'Тип': 'Непрерывная',  'Описание': 'Возраст на момент диагноза (лет)'},
    {'Переменная': 'sex',              'Тип': 'Бинарная',     'Описание': '1=мужчина, 0=женщина'},
    {'Переменная': 'hr',               'Тип': 'Непрерывная',  'Описание': 'ЧСС (уд/мин)'},
    {'Переменная': 'lvef_cat_num',     'Тип': 'Порядковая',   'Описание': 'ФВ ЛЖ числом: 0=норма, 1=средне, 2=плохо'},
    {'Переменная': 'la_size',          'Тип': 'Непрерывная',  'Описание': 'Размер левого предсердия (мм)'},
    {'Переменная': 'esr_start',        'Тип': 'Непрерывная',  'Описание': 'СОЭ начальное (мм/ч), winsorized'},
    {'Переменная': 'crp_start',        'Тип': 'Непрерывная',  'Описание': 'СРБ начальный (мг/л), winsorized'},
    {'Переменная': 'wbc_start',        'Тип': 'Непрерывная',  'Описание': 'Лейкоциты начальные (×10⁹/л), winsorized'},
    {'Переменная': 'bmi',              'Тип': 'Непрерывная',  'Описание': 'ИМТ (кг/м²)'},
    {'Переменная': 'hgb_start',        'Тип': 'Непрерывная',  'Описание': 'Гемоглобин начальный (г/л)'},
    {'Переменная': 'anemia',           'Тип': 'Бинарная',     'Описание': '1=анемия (Hb ниже нормы по полу)'},
    {'Переменная': 'glucose_start',    'Тип': 'Непрерывная',  'Описание': 'Глюкоза начальная (ммоль/л), winsorized'},
    {'Переменная': 'chol_start',       'Тип': 'Непрерывная',  'Описание': 'Холестерин начальный (ммоль/л)'},
    {'Переменная': 'qtc_start_ms',     'Тип': 'Непрерывная',  'Описание': 'QTc исходный (мс)'},
    {'Переменная': 'delta_qtc_ms',     'Тип': 'Непрерывная',  'Описание': 'ΔQTc = QTc_последнее − QTc_начало (мс)'},
    {'Переменная': 'delta_qtc_gt30',   'Тип': 'Бинарная',     'Описание': '1=ΔQTc ≥ 30 мс (настороженность)'},
    {'Переменная': 'delta_qtc_gt60',   'Тип': 'Бинарная',     'Описание': '1=ΔQTc ≥ 60 мс (коррекция терапии)'},
    {'Переменная': 'stage_advanced',   'Тип': 'Бинарная',     'Описание': '1=стадия III–IV'},
    {'Переменная': 'histology',        'Тип': 'Порядковая',   'Описание': 'Гистология: 1=G1, 2=G2, 3=G3'},
    {'Переменная': 'mets_liver',       'Тип': 'Бинарная',     'Описание': '1=метастазы в печень'},
    {'Переменная': 'cht_scheme',       'Тип': 'Категориальная','Описание': '0=нет ХТ, 1=FOLFOX, 2=FOLFIRI, 3=XELOX, 4=DeGramont, 5=Другое'},
    {'Переменная': 'cht_scheme_label',  'Тип': 'Строковая',    'Описание': 'Название схемы ХТ (для графиков и таблиц)'},
    {'Переменная': 'platinum',          'Тип': 'Бинарная',     'Описание': '1=платиносодержащие препараты'},
    {'Переменная': 'n_courses',        'Тип': 'Непрерывная',  'Описание': 'Количество курсов ХТ'},
    {'Переменная': 'bevacizumab',      'Тип': 'Бинарная',     'Описание': '1=получал бевацизумаб'},
    {'Переменная': 'cvd_history',      'Тип': 'Бинарная',     'Описание': '1=ССЗ в анамнезе'},
])
var_dict.to_csv(OUTPUT_DIR / 'РТК_ССО_словарь_переменных_v4.csv', index=False, encoding='utf-8-sig')
print('Сохранён: РТК_ССО_словарь_переменных_v4.csv')

print('\n=== Подготовка данных завершена (v4) ===')
print(f'  Пациентов:                {len(df_final)}')
print(f'  Столбцов итого:           {df_final.shape[1]}')
print(f'  Предикторов:              {len(PREDICTORS)}')
print(f'  Кардиотоксичность:        {df_final["event_cardiotox"].sum()} ({df_final["event_cardiotox"].mean()*100:.1f}%)')
print(f'  Смерть без кардиотоксичности:{df_final["event_death"].sum()}')
print(f'  Медиана наблюдения:       {df_final["time_days"].median():.0f} дней')
print(f'\n  Следующий шаг:  Kaplan–Meier + Cox regression + Aalen–Johansen CIF (lifelines)')